In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd


# The notebook is saved directly inside the Code folder.
NOTEBOOK_DIRECTORY = Path.cwd().resolve()
PROJECT_ROOT = NOTEBOOK_DIRECTORY.parent

INPUT_FILE = (
    PROJECT_ROOT
    / "Data"
    / "Processed"
    / "analysis_cohort.xlsx"
)

PROCESSED_DATA_DIRECTORY = (
    PROJECT_ROOT
    / "Data"
    / "Processed"
)


print("Notebook directory:")
print(NOTEBOOK_DIRECTORY)

print("\nProject root:")
print(PROJECT_ROOT)

print("\nInput dataset:")
print(INPUT_FILE)


assert INPUT_FILE.exists(), (
    f"The analysis cohort was not found:\n{INPUT_FILE}"
)

In [ ]:
analysis_data_original = pd.read_excel(
    INPUT_FILE,
    sheet_name="Analysis_Cohort",
    dtype=object
)


print(f"Number of patients: {len(analysis_data_original):,}")
print(f"Number of columns: {analysis_data_original.shape[1]}")


assert len(analysis_data_original) == 1370, (
    "Expected 1,370 patients, but found "
    f"{len(analysis_data_original):,}."
)


outcome_counts = (
    analysis_data_original[
        "Five_Year_Recurrence"
    ]
    .value_counts(dropna=False)
)


display(outcome_counts)


assert outcome_counts.get("Yes", 0) == 73
assert outcome_counts.get("No", 0) == 1297

print("The strict analysis cohort was loaded successfully.")

In [ ]:
analysis_data = analysis_data_original.copy()


original_column_names = list(
    analysis_data.columns
)

cleaned_column_names = [
    str(column).strip()
    for column in original_column_names
]


column_name_changes = pd.DataFrame({
    "Original_Column_Name": original_column_names,
    "Cleaned_Column_Name": cleaned_column_names
})


column_name_changes = column_name_changes.loc[
    column_name_changes[
        "Original_Column_Name"
    ].ne(
        column_name_changes[
            "Cleaned_Column_Name"
        ]
    )
]


analysis_data.columns = cleaned_column_names


assert not analysis_data.columns.duplicated().any(), (
    "Removing spaces created duplicate column names."
)


print("Column names changed:")
display(column_name_changes)


print("\nCurrent column names:")

for number, column in enumerate(
    analysis_data.columns,
    start=1
):
    print(f"{number:2}. {column}")

In [ ]:
variable_inventory = pd.DataFrame({
    "Variable_Number": range(
        1,
        len(analysis_data.columns) + 1
    ),
    "Variable_Name": analysis_data.columns,
    "Imported_Data_Type": [
        str(data_type)
        for data_type in analysis_data.dtypes
    ]
})


display(variable_inventory)

In [ ]:
non_predictor_columns = [
    "Source_Excel_Row",
    "Diagnosis_Date_Used",
    "Diagnosis_Date_Quality",
    "Recurrence_Date_Used",
    "Recurrence_Date_Quality",
    "Five_Year_Anniversary",
    "Administrative_Study_End",
    "Eligible_For_5Y_Analysis",
    "Cohort_Decision_Reason",
    "Year First Seen",
    "Date of Diagnosis",
    "Vital Status",
    "Date Recurrence",
    "Recurrence"
]


target_column = "Five_Year_Recurrence"


missing_non_predictors = [
    column
    for column in non_predictor_columns
    if column not in analysis_data.columns
]


if missing_non_predictors:
    print(
        "These expected non-predictor variables "
        "were not found:"
    )

    print(missing_non_predictors)


candidate_predictor_columns = [
    column
    for column in analysis_data.columns
    if column not in non_predictor_columns
    and column != target_column
]


print(
    "Number of candidate predictors:",
    len(candidate_predictor_columns)
)

print("\nCandidate predictors:")

for number, column in enumerate(
    candidate_predictor_columns,
    start=1
):
    print(f"{number:2}. {column}")

In [ ]:
missingness_summary = pd.DataFrame({
    "Variable": candidate_predictor_columns,

    "Missing_Count": [
        analysis_data[column].isna().sum()
        for column in candidate_predictor_columns
    ]
})


missingness_summary["Missing_Percentage"] = (
    missingness_summary["Missing_Count"]
    / len(analysis_data)
    * 100
).round(2)


missingness_summary = (
    missingness_summary
    .sort_values(
        by=[
            "Missing_Percentage",
            "Variable"
        ],
        ascending=[False, True]
    )
    .reset_index(drop=True)
)


display(missingness_summary)

In [ ]:
special_labels = [
    "NONE",
    "(NONE)",
    "UNKNOWN",
    "NOT DOCUMENTED IN MEDICAL RECORD",
    "TEST DONE BUT RESULTS NOT IN MEDICAL RECORD",
    ""
]


special_value_records = []


for column in candidate_predictor_columns:

    normalized_values = (
        analysis_data[column]
        .astype("string")
        .str.strip()
        .str.upper()
    )

    for label in special_labels:

        count = int(
            normalized_values.eq(label).sum()
        )

        if count > 0:
            special_value_records.append({
                "Variable": column,
                "Special_Value": (
                    "[BLANK TEXT]"
                    if label == ""
                    else label
                ),
                "Count": count,
                "Percentage": round(
                    count / len(analysis_data) * 100,
                    2
                )
            })


special_values_summary = pd.DataFrame(
    special_value_records
)


if len(special_values_summary) > 0:

    special_values_summary = (
        special_values_summary
        .sort_values(
            by=["Variable", "Count"],
            ascending=[True, False]
        )
        .reset_index(drop=True)
    )

    display(special_values_summary)

else:
    print("No special text categories were found.")

In [ ]:
expected_numeric_variables = [
    "Age at Diagnosis",
    "Tumor Size",
    "Parity",
    "Number of Abortion",
    "Age at First Child Birth",
    "Menarche Age"
]


numeric_audit_records = []
nonnumeric_value_records = []


for column in expected_numeric_variables:

    if column not in analysis_data.columns:
        continue

    original_values = analysis_data[column]

    numeric_values = pd.to_numeric(
        original_values,
        errors="coerce"
    )

    original_not_missing = (
        original_values.notna()
        & original_values.astype(str).str.strip().ne("")
    )

    could_not_convert = (
        original_not_missing
        & numeric_values.isna()
    )

    nonnumeric_values = (
        original_values.loc[could_not_convert]
        .astype(str)
        .str.strip()
        .value_counts()
    )

    numeric_audit_records.append({
        "Variable": column,
        "Valid_Numeric_Count": (
            numeric_values.notna().sum()
        ),
        "Missing_or_Nonnumeric_Count": (
            numeric_values.isna().sum()
        ),
        "Minimum": numeric_values.min(),
        "Maximum": numeric_values.max(),
        "Mean": numeric_values.mean(),
        "Median": numeric_values.median()
    })

    for value, count in nonnumeric_values.items():
        nonnumeric_value_records.append({
            "Variable": column,
            "Nonnumeric_Value": value,
            "Count": count
        })


numeric_audit_summary = pd.DataFrame(
    numeric_audit_records
)

nonnumeric_values_summary = pd.DataFrame(
    nonnumeric_value_records
)


print("Numerical-variable summary:")
display(numeric_audit_summary)


print("\nNonnumeric entries found in expected numerical variables:")

if len(nonnumeric_values_summary) > 0:
    display(nonnumeric_values_summary)
else:
    print("None")

In [ ]:
def show_category_counts(column):
    """
    Display all observed values and counts for one variable.
    """

    if column not in analysis_data.columns:
        raise KeyError(
            f"{column!r} was not found."
        )

    category_table = (
        analysis_data[column]
        .fillna("[ACTUAL MISSING]")
        .astype(str)
        .str.strip()
        .replace("", "[BLANK TEXT]")
        .value_counts(dropna=False)
        .rename_axis("Observed_Value")
        .reset_index(name="Count")
    )

    category_table["Percentage"] = (
        category_table["Count"]
        / len(analysis_data)
        * 100
    ).round(2)

    print(f"Variable: {column}")

    display(category_table)

    return category_table

In [ ]:
show_category_counts("Marital Status at DX")

In [ ]:
show_category_counts("Gender")
show_category_counts("Nationality")
show_category_counts("Patient History, Tobacco")
show_category_counts("Family history (1st) this cancer")
show_category_counts("Family history (1st) any cancer")
show_category_counts("Family history (2nd) this cancer")
show_category_counts("Family history (2nd) any cancer")

In [ ]:
remaining_categorical_variables = [
    "Marital Status at DX",
    "Site - Primary (ICD-O-3)",
    "Laterality",
    "Histology/Behavior (ICD-O-3)",
    "Grade Clinical",
    "Grade Pathological",
    "Summary Stage",
    "Estrogen Receptor",
    "Progesterone Receptor",
    "HER2 Overall",
    "Distant Metastasis Site",
    "Surgery",
    "Chemotherapy",
    "Hormone",
    "Pregnant at Diagnosis",
    "Menopause Status (Y/N, UNKNOWN)"
]


for column in remaining_categorical_variables:

    print("\n" + "=" * 80)

    show_category_counts(column)

In [ ]:
treatment_variables = [
    "Surgery",
    "Chemotherapy",
    "Hormone"
]


treatment_summary_records = []


for column in treatment_variables:

    values = (
        analysis_data[column]
        .astype("string")
        .str.strip()
        .str.upper()
    )

    summary = {
        "Variable": column,
        "Yes": int(values.eq("YES").sum()),
        "Explicit_No": int(values.eq("NO").sum()),
        "Blank_or_NaN": int(
            analysis_data[column].isna().sum()
            + values.eq("").sum()
        )
    }

    summary["Total"] = (
        summary["Yes"]
        + summary["Explicit_No"]
        + summary["Blank_or_NaN"]
    )

    treatment_summary_records.append(summary)


treatment_summary = pd.DataFrame(
    treatment_summary_records
)


display(treatment_summary)


assert treatment_summary["Total"].eq(
    len(analysis_data)
).all()

In [ ]:
numerical_boundary_checks = {
    "Age at Diagnosis": {
        "condition": (
            pd.to_numeric(
                analysis_data["Age at Diagnosis"],
                errors="coerce"
            ) > 90
        )
    },

    "Tumor Size": {
        "condition": (
            pd.to_numeric(
                analysis_data["Tumor Size"],
                errors="coerce"
            ).le(0)
            |
            pd.to_numeric(
                analysis_data["Tumor Size"],
                errors="coerce"
            ).gt(200)
        )
    }
}


for variable, specification in numerical_boundary_checks.items():

    print("\n" + "=" * 80)
    print(f"Boundary review: {variable}")

    review_rows = analysis_data.loc[
        specification["condition"],
        [
            "Source_Excel_Row",
            "Age at Diagnosis",
            "Tumor Size",
            "Summary Stage",
            "Histology/Behavior (ICD-O-3)",
            "Five_Year_Recurrence"
        ]
    ]

    display(review_rows)

In [ ]:
# Preserve the audited cohort unchanged.
cleaned_data = analysis_data.copy()


# Recode treatment blanks as No, based on the hospital
# registry coding convention confirmed by the contributor.
treatment_variables = [
    "Surgery",
    "Chemotherapy",
    "Hormone"
]


for column in treatment_variables:

    cleaned_data[column] = (
        cleaned_data[column]
        .astype("string")
        .str.strip()
        .str.upper()
        .replace("", pd.NA)
        .fillna("NO")
        .map({
            "YES": "Yes",
            "NO": "No"
        })
    )


print("Recoded treatment distributions:")

for column in treatment_variables:

    print(f"\n{column}")

    display(
        cleaned_data[column]
        .value_counts(dropna=False)
    )

In [ ]:
def grade_available(series):

    normalized = (
        series
        .astype("string")
        .str.strip()
        .str.upper()
    )

    unavailable = normalized.isin([
        "(NONE)",
        "GRADE CANNOT BE ASSESSED (GX)"
    ]) | normalized.isna()

    return ~unavailable


clinical_grade_available = grade_available(
    analysis_data["Grade Clinical"]
)

pathological_grade_available = grade_available(
    analysis_data["Grade Pathological"]
)


grade_availability_summary = pd.DataFrame({
    "Clinical_Grade_Available": (
        clinical_grade_available
    ),
    "Pathological_Grade_Available": (
        pathological_grade_available
    )
})


display(
    grade_availability_summary
    .value_counts()
    .rename("Patients")
    .reset_index()
)


print(
    "Patients with at least one usable grade:",
    int(
        (
            clinical_grade_available
            | pathological_grade_available
        ).sum()
    )
)

print(
    "Patients with neither grade available:",
    int(
        ~(
            clinical_grade_available
            | pathological_grade_available
        ).sum()
    )
)

In [ ]:
display(
    pd.crosstab(
        analysis_data["Gender"],
        analysis_data[
            "Menopause Status (Y/N, UNKNOWN)"
        ].fillna("[MISSING]"),
        margins=True
    )
)

In [ ]:
metastasis_present = (
    analysis_data[
        "Distant Metastasis Site"
    ]
    .astype("string")
    .str.strip()
    .str.upper()
    .ne("NONE")
)


metastasis_stage_check = pd.crosstab(
    analysis_data["Summary Stage"],
    metastasis_present.map({
        True: "Metastasis site recorded",
        False: "No metastasis site recorded"
    }),
    margins=True
)


display(metastasis_stage_check)

In [ ]:
tumor_size_numeric = pd.to_numeric(
    analysis_data["Tumor Size"],
    errors="coerce"
)


extreme_tumor_records = analysis_data.loc[
    tumor_size_numeric.le(0)
    | tumor_size_numeric.gt(200),
    [
        "Source_Excel_Row",
        "Age at Diagnosis",
        "Tumor Size",
        "Summary Stage",
        "Distant Metastasis Site",
        "Histology/Behavior (ICD-O-3)",
        "Five_Year_Recurrence"
    ]
]


display(extreme_tumor_records)

In [ ]:
grade_available_any = (
    clinical_grade_available
    | pathological_grade_available
)

grade_unavailable_both = ~grade_available_any


print(
    "Patients with at least one usable grade:",
    int(grade_available_any.sum())
)

print(
    "Patients with neither grade available:",
    int(grade_unavailable_both.sum())
)

In [ ]:
def recode_grade(value):
    """
    Recode the original grade descriptions into:
    Grade 1, Grade 2, Grade 3, or Unknown.
    """

    if pd.isna(value):
        return "Unknown"

    text = str(value).strip().upper()

    unavailable_values = {
        "",
        "(NONE)",
        "GRADE CANNOT BE ASSESSED (GX)"
    }

    if text in unavailable_values:
        return "Unknown"

    if (
        text.startswith("G1:")
        or "NUCLEAR GRADE I " in text
        or text == "WELL DIFFERENTIATED"
    ):
        return "Grade 1"

    if (
        text.startswith("G2:")
        or "NUCLEAR GRADE II " in text
        or text == "MODERATELY DIFFERENTIATED"
    ):
        return "Grade 2"

    if (
        text.startswith("G3:")
        or "NUCLEAR GRADE III " in text
        or text == "POORLY DIFFERENTIATED"
    ):
        return "Grade 3"

    return "Unknown"


cleaned_data["Clinical_Grade_Clean"] = (
    cleaned_data["Grade Clinical"]
    .apply(recode_grade)
)

cleaned_data["Pathological_Grade_Clean"] = (
    cleaned_data["Grade Pathological"]
    .apply(recode_grade)
)


print("Cleaned clinical-grade distribution:")

display(
    cleaned_data[
        "Clinical_Grade_Clean"
    ].value_counts(dropna=False)
)


print("\nCleaned pathological-grade distribution:")

display(
    cleaned_data[
        "Pathological_Grade_Clean"
    ].value_counts(dropna=False)
)

In [ ]:
# =========================================================
# Compare clinical and pathological grades
# =========================================================

# Select patients with both grades available.
both_grades_available = (
    cleaned_data["Clinical_Grade_Clean"].ne("Unknown")
    &
    cleaned_data["Pathological_Grade_Clean"].ne("Unknown")
)


grade_comparison_data = cleaned_data.loc[
    both_grades_available,
    [
        "Source_Excel_Row",
        "Clinical_Grade_Clean",
        "Pathological_Grade_Clean",
        "Five_Year_Recurrence"
    ]
].copy()


# Determine whether the two grades agree.
grade_comparison_data["Grades_Agree"] = (
    grade_comparison_data["Clinical_Grade_Clean"]
    ==
    grade_comparison_data["Pathological_Grade_Clean"]
)


# Create the cross-tabulation.
grade_comparison_table = pd.crosstab(
    grade_comparison_data["Clinical_Grade_Clean"],
    grade_comparison_data["Pathological_Grade_Clean"],
    margins=True
)


# Calculate agreement statistics.
number_with_both_grades = len(
    grade_comparison_data
)

number_agreeing = int(
    grade_comparison_data["Grades_Agree"].sum()
)

number_disagreeing = int(
    (~grade_comparison_data["Grades_Agree"]).sum()
)

agreement_percentage = round(
    number_agreeing
    / number_with_both_grades
    * 100,
    2
)


print("Clinical versus pathological grade:")
display(grade_comparison_table)


print(
    "\nPatients with both grades available:",
    number_with_both_grades
)

print(
    "Patients whose grades agree:",
    number_agreeing
)

print(
    "Patients whose grades disagree:",
    number_disagreeing
)

print(
    "Agreement percentage:",
    f"{agreement_percentage}%"
)


# Verify that the counts reconcile.
assert (
    number_agreeing + number_disagreeing
    == number_with_both_grades
), "Grade agreement counts do not reconcile."


# Display patients with disagreement.
grade_disagreement_records = (
    grade_comparison_data.loc[
        ~grade_comparison_data["Grades_Agree"]
    ]
    .sort_values(
        by=[
            "Clinical_Grade_Clean",
            "Pathological_Grade_Clean",
            "Source_Excel_Row"
        ]
    )
    .reset_index(drop=True)
)


print(
    "\nNumber of grade-disagreement records:",
    len(grade_disagreement_records)
)

display(grade_disagreement_records)

In [ ]:
# =========================================================
# Create the combined tumour-grade variable
# =========================================================

cleaned_data["Combined_Grade"] = np.where(
    cleaned_data["Pathological_Grade_Clean"].ne("Unknown"),
    cleaned_data["Pathological_Grade_Clean"],
    cleaned_data["Clinical_Grade_Clean"]
)


combined_grade_summary = (
    cleaned_data["Combined_Grade"]
    .value_counts(dropna=False)
    .rename_axis("Combined_Grade")
    .reset_index(name="Count")
)


combined_grade_summary["Percentage"] = (
    combined_grade_summary["Count"]
    / len(cleaned_data)
    * 100
).round(2)


display(combined_grade_summary)


# Validation checks
assert (
    cleaned_data["Combined_Grade"]
    .ne("Unknown")
    .sum()
    == 510
), "Expected 510 patients with an available combined grade."


assert (
    cleaned_data["Combined_Grade"]
    .eq("Unknown")
    .sum()
    == 860
), "Expected 860 patients with unknown combined grade."


print(
    "Combined grade created successfully."
)

print(
    "Patients with usable combined grade:",
    cleaned_data["Combined_Grade"]
    .ne("Unknown")
    .sum()
)

print(
    "Patients with unknown combined grade:",
    cleaned_data["Combined_Grade"]
    .eq("Unknown")
    .sum()
)

In [ ]:
# =========================================================
# Audit overlap among the four family-history variables
# =========================================================

family_history_columns = {
    "First_Degree_Breast": (
        "Family history (1st) this cancer"
    ),

    "First_Degree_Any_Cancer": (
        "Family history (1st) any cancer"
    ),

    "Second_Degree_Breast": (
        "Family history (2nd) this cancer"
    ),

    "Second_Degree_Any_Cancer": (
        "Family history (2nd) any cancer"
    )
}


def simplify_family_history(value):
    """
    Temporarily classify a family-history field as:
    Yes, No, or Unknown.
    """

    if pd.isna(value):
        return "Unknown"

    text = str(value).strip().upper()

    if text in {"", "UNKNOWN"}:
        return "Unknown"

    if text == "NONE":
        return "No"

    if "RELATIVE" in text:
        return "Yes"

    return "Unknown"


family_history_audit = pd.DataFrame({
    "Source_LongLeg": analysis_data[
        "Source_Excel_Row"
    ]
})


for new_name, original_name in (
    family_history_columns.items()
):

    family_history_audit[new_name] = (
        analysis_data[original_name]
        .apply(simplify_family_history)
    )


family_history_audit[
    "Five_Year_Recurrence"
] = analysis_data[
    "Five_Year_Recurrence"
]


print("Simplified distributions:")

for column in family_history_columns:

    print(f"\n{column}")

    display(
        family_history_audit[column]
        .value_counts(dropna=False)
    )

In [ ]:
print(
    "First-degree versus second-degree "
    "breast cancer family history:"
)

breast_history_by_degree = pd.crosstab(
    family_history_audit[
        "First_Degree_Breast"
    ],
    family_history_audit[
        "Second_Degree_Breast"
    ],
    margins=True
)

display(breast_history_by_degree)

In [ ]:
print(
    "First-degree versus second-degree "
    "breast cancer family history:"
)

breast_history_by_degree = pd.crosstab(
    family_history_audit[
        "First_Degree_Breast"
    ],
    family_history_audit[
        "Second_Degree_Breast"
    ],
    margins=True
)

display(breast_history_by_degree)

In [ ]:
# First-degree comparison
first_degree_overlap = pd.crosstab(
    family_history_audit["First_Degree_Breast"],
    family_history_audit["First_Degree_Any_Cancer"],
    margins=True
)

print(
    "FIRST-DEGREE BREAST CANCER "
    "VERSUS FIRST-DEGREE ANY CANCER"
)

display(first_degree_overlap)


# Second-degree comparison
second_degree_overlap = pd.crosstab(
    family_history_audit["Second_Degree_Breast"],
    family_history_audit["Second_Degree_Any_Cancer"],
    margins=True
)

print(
    "\nSECOND-DEGREE BREAST CANCER "
    "VERSUS SECOND-DEGREE ANY CANCER"
)

display(second_degree_overlap)

In [ ]:
# =========================================================
# Compare any-cancer history across family-degree groups
# =========================================================

any_cancer_by_degree = pd.crosstab(
    family_history_audit[
        "First_Degree_Any_Cancer"
    ],
    family_history_audit[
        "Second_Degree_Any_Cancer"
    ],
    margins=True
)


print(
    "FIRST-DEGREE VERSUS SECOND-DEGREE "
    "ANY-CANCER FAMILY HISTORY"
)

display(any_cancer_by_degree)

In [ ]:
# =========================================================
# Create the two final family-history variables
# =========================================================

def combine_family_history(first_degree, second_degree):
    """
    Combine first- and second-degree family-history records.

    Yes takes priority because family history is confirmed.
    Unknown is used when no positive history is recorded
    but at least one source field is unknown.
    No is used only when both fields are No.
    """

    values = {
        first_degree,
        second_degree
    }

    if "Yes" in values:
        return "Yes"

    if "Unknown" in values:
        return "Unknown"

    return "No"


# Breast cancer family history across either degree.
cleaned_data[
    "Family_History_Breast_Cancer"
] = family_history_audit.apply(
    lambda row: combine_family_history(
        row["First_Degree_Breast"],
        row["Second_Degree_Breast"]
    ),
    axis=1
)


# Other/any-cancer family history across either degree.
cleaned_data[
    "Family_History_Other_Cancer"
] = family_history_audit.apply(
    lambda row: combine_family_history(
        row["First_Degree_Any_Cancer"],
        row["Second_Degree_Any_Cancer"]
    ),
    axis=1
)


print("Breast cancer family-history distribution:")

display(
    cleaned_data[
        "Family_History_Breast_Cancer"
    ].value_counts(dropna=False)
)


print("\nOther cancer family-history distribution:")

display(
    cleaned_data[
        "Family_History_Other_Cancer"
    ].value_counts(dropna=False)
)

In [ ]:
breast_family_history_outcome = pd.crosstab(
    cleaned_data["Family_History_Breast_Cancer"],
    cleaned_data["Five_Year_Recurrence"],
    margins=True
)

other_family_history_outcome = pd.crosstab(
    cleaned_data["Family_History_Other_Cancer"],
    cleaned_data["Five_Year_Recurrence"],
    margins=True
)


print(
    "Breast cancer family history "
    "by five-year recurrence:"
)

display(breast_family_history_outcome)


print(
    "\nOther cancer family history "
    "by five-year recurrence:"
)

display(other_family_history_outcome)

In [ ]:
# =========================================================
# COMPLETE PREDICTOR CLEANING AND RECODING
# =========================================================

import re
cleaned_data = analysis_data.copy()


# ---------------------------------------------------------
# 1. Basic helper functions
# ---------------------------------------------------------

def normalize_text(value):

    if pd.isna(value):
        return None

    text = str(value).strip()

    if text == "":
        return None

    return text.upper()


def simplify_family_history(value):

    text = normalize_text(value)

    if text is None or text == "UNKNOWN":
        return "Unknown"

    if text == "NONE":
        return "No"

    if "RELATIVE" in text:
        return "Yes"

    return "Unknown"


def combine_family_history(first_degree, second_degree):

    values = {first_degree, second_degree}

    if "Yes" in values:
        return "Yes"

    if "Unknown" in values:
        return "Unknown"

    return "No"


# ---------------------------------------------------------
# 2. Age at diagnosis
# ---------------------------------------------------------

cleaned_data["Age_at_Diagnosis"] = pd.to_numeric(
    cleaned_data["Age at Diagnosis"],
    errors="coerce"
)

# Ages 101 and 103 are retained as reported.
cleaned_data["Age_Over_100_Flag"] = np.where(
    cleaned_data["Age_at_Diagnosis"].gt(100),
    "Yes",
    "No"
)


# ---------------------------------------------------------
# 3. Marital status
# ---------------------------------------------------------

marital_mapping = {
    "MARRIED": "Married",
    "SINGLE": "Single",
    "WIDOWED": "Widowed",
    "DIVORCED": "Divorced",
    "UNKNOWN": "Unknown"
}

cleaned_data["Marital_Status"] = (
    cleaned_data["Marital Status at DX"]
    .apply(normalize_text)
    .map(marital_mapping)
    .fillna("Unknown")
)


# ---------------------------------------------------------
# 4. Gender and nationality
# ---------------------------------------------------------

cleaned_data["Gender_Clean"] = (
    cleaned_data["Gender"]
    .apply(normalize_text)
    .map({
        "FEMALE": "Female",
        "MALE": "Male"
    })
    .fillna("Unknown")
)


cleaned_data["Nationality_Clean"] = (
    cleaned_data["Nationality"]
    .apply(normalize_text)
    .map({
        "SAUDI": "Saudi",
        "NON SAUDI": "Non_Saudi"
    })
    .fillna("Unknown")
)


# ---------------------------------------------------------
# 5. Tobacco history
# ---------------------------------------------------------

def recode_tobacco(value):

    text = normalize_text(value)

    if text is None or text == "UNKNOWN":
        return "Unknown"

    if text == "NEVER USED":
        return "Never"

    if "CURRENT" in text or text == "CIGARETTE SMOKER CURRENT":
        return "Ever"

    if text == "NEVER SMOKED":
        return "Never"

    if "PREVIOUS" in text or "FORMER" in text:
        return "Ever"

    return "Unknown"


cleaned_data["Tobacco_History_Clean"] = (
    cleaned_data["Patient History, Tobacco"]
    .apply(recode_tobacco)
)


# ---------------------------------------------------------
# 6. Family history
# ---------------------------------------------------------

first_breast = cleaned_data[
    "Family history (1st) this cancer"
].apply(simplify_family_history)

second_breast = cleaned_data[
    "Family history (2nd) this cancer"
].apply(simplify_family_history)

first_other = cleaned_data[
    "Family history (1st) any cancer"
].apply(simplify_family_history)

second_other = cleaned_data[
    "Family history (2nd) any cancer"
].apply(simplify_family_history)


cleaned_data["Family_History_Breast_Cancer"] = [
    combine_family_history(first, second)
    for first, second in zip(
        first_breast,
        second_breast
    )
]


cleaned_data["Family_History_Other_Cancer"] = [
    combine_family_history(first, second)
    for first, second in zip(
        first_other,
        second_other
    )
]


# ---------------------------------------------------------
# 7. Primary tumour site
# ---------------------------------------------------------

def recode_primary_site(value):

    text = normalize_text(value)

    if text is None:
        return "Unknown"

    if text.startswith(("C504", "C505")):
        return "Outer_Quadrant"

    if text.startswith(("C502", "C503")):
        return "Inner_Quadrant"

    if text.startswith(("C500", "C501")):
        return "Central_or_Nipple"

    if text.startswith("C508"):
        return "Overlapping_Lesion"

    if text.startswith("C509"):
        return "Breast_NOS"

    return "Other"


cleaned_data["Primary_Site_Group"] = (
    cleaned_data["Site - Primary (ICD-O-3)"]
    .apply(recode_primary_site)
)


# ---------------------------------------------------------
# 8. Laterality
# ---------------------------------------------------------

cleaned_data["Laterality_Clean"] = (
    cleaned_data["Laterality"]
    .apply(normalize_text)
    .map({
        "LEFT": "Left",
        "RIGHT": "Right",
        "PAIRED NO INFO": "Unknown"
    })
    .fillna("Unknown")
)


# ---------------------------------------------------------
# 9. Histology
# ---------------------------------------------------------

def recode_histology(value):

    text = normalize_text(value)

    if text is None:
        return "Unknown"

    code_match = re.match(r"(\d{5})", text)

    if code_match is None:
        return "Other"

    code = code_match.group(1)

    if code == "85003":
        return "Invasive_Ductal_NST"

    if code == "85203":
        return "Invasive_Lobular"

    if code in {"85223", "85233", "85243"}:
        return "Mixed_Invasive"

    # Final digit 2 indicates in situ behaviour.
    if code.endswith("2"):
        return "In_Situ"

    return "Other_Invasive"


cleaned_data["Histology_Group"] = (
    cleaned_data["Histology/Behavior (ICD-O-3)"]
    .apply(recode_histology)
)


# ---------------------------------------------------------
# 10. Clinical and pathological grade
# ---------------------------------------------------------

def recode_grade(value):

    text = normalize_text(value)

    if text in {
        None,
        "(NONE)",
        "GRADE CANNOT BE ASSESSED (GX)"
    }:
        return "Unknown"

    if (
        text.startswith("G1:")
        or "NUCLEAR GRADE I " in text
        or text == "WELL DIFFERENTIATED"
    ):
        return "Grade_1"

    if (
        text.startswith("G2:")
        or "NUCLEAR GRADE II " in text
        or text == "MODERATELY DIFFERENTIATED"
    ):
        return "Grade_2"

    if (
        text.startswith("G3:")
        or "NUCLEAR GRADE III " in text
        or text == "POORLY DIFFERENTIATED"
    ):
        return "Grade_3"

    return "Unknown"


cleaned_data["Clinical_Grade_Clean"] = (
    cleaned_data["Grade Clinical"]
    .apply(recode_grade)
)

cleaned_data["Pathological_Grade_Clean"] = (
    cleaned_data["Grade Pathological"]
    .apply(recode_grade)
)


cleaned_data["Combined_Grade"] = np.where(
    cleaned_data["Pathological_Grade_Clean"].ne("Unknown"),
    cleaned_data["Pathological_Grade_Clean"],
    cleaned_data["Clinical_Grade_Clean"]
)


# ---------------------------------------------------------
# 11. Tumour size
# ---------------------------------------------------------

cleaned_data["Tumor_Size_mm"] = pd.to_numeric(
    cleaned_data["Tumor Size"],
    errors="coerce"
)

# Two zero values become missing.
cleaned_data.loc[
    cleaned_data["Tumor_Size_mm"].le(0),
    "Tumor_Size_mm"
] = np.nan

# Values 230 and 270 are retained but flagged.
cleaned_data["Tumor_Size_Over_200_Flag"] = np.where(
    cleaned_data["Tumor_Size_mm"].gt(200),
    "Yes",
    "No"
)


# ---------------------------------------------------------
# 12. Summary stage
# ---------------------------------------------------------

def recode_summary_stage(value):

    text = normalize_text(value)

    if text is None:
        return "Unknown"

    if "IN SITU" in text:
        return "In_Situ"

    if "LOCALIZED" in text:
        return "Localized"

    if "DISTANT" in text:
        return "Distant"

    if "REGIONAL" in text:
        return "Regional"

    return "Unknown"


cleaned_data["Summary_Stage_Clean"] = (
    cleaned_data["Summary Stage"]
    .apply(recode_summary_stage)
)


# ---------------------------------------------------------
# 13. ER and PR status
# ---------------------------------------------------------

def recode_receptor(value, receptor):

    text = normalize_text(value)

    unavailable_values = {
        None,
        "(NONE)",
        "NOT DOCUMENTED IN MEDICAL RECORD",
        "TEST DONE, RESULTS NOT IN CHART",
        "TEST DONE BUT RESULTS NOT IN MEDICAL RECORD"
    }

    if text in unavailable_values:
        return "Unknown"

    if "POSITIVE" in text:
        return "Positive"

    if "NEGATIVE" in text:
        return "Negative"

    return "Unknown"


cleaned_data["ER_Status"] = (
    cleaned_data["Estrogen Receptor"]
    .apply(lambda value: recode_receptor(value, "ER"))
)

cleaned_data["PR_Status"] = (
    cleaned_data["Progesterone Receptor"]
    .apply(lambda value: recode_receptor(value, "PR"))
)


# ---------------------------------------------------------
# 14. HER2 status
# ---------------------------------------------------------

def recode_her2(value):

    text = normalize_text(value)

    unavailable_values = {
        None,
        "(NONE)",
        "NOT DOCUMENTED IN MEDICAL RECORD",
        "TEST DONE, RESULTS NOT IN CHART",
        "TEST DONE BUT RESULTS NOT IN MEDICAL RECORD"
    }

    if text in unavailable_values:
        return "Unknown"

    if "POSITIVE" in text:
        return "Positive"

    if "NEGATIVE" in text or "EQUIVOCAL" in text:
        return "Negative_or_Equivocal"

    return "Unknown"


cleaned_data["HER2_Status"] = (
    cleaned_data["HER2 Overall"]
    .apply(recode_her2)
)


# ---------------------------------------------------------
# 15. Distant metastasis variables
# ---------------------------------------------------------

metastasis_site_normalized = (
    cleaned_data["Distant Metastasis Site"]
    .apply(normalize_text)
)


cleaned_data["Distant_Metastasis_Recorded"] = np.where(
    metastasis_site_normalized.eq("NONE"),
    "No",
    "Yes"
)


stage_indicates_distant = (
    cleaned_data["Summary_Stage_Clean"]
    .eq("Distant")
)

site_indicates_metastasis = (
    cleaned_data["Distant_Metastasis_Recorded"]
    .eq("Yes")
)


cleaned_data["Stage_Metastasis_Disagreement"] = np.where(
    stage_indicates_distant != site_indicates_metastasis,
    "Yes",
    "No"
)


# ---------------------------------------------------------
# 16. Treatment variables
# ---------------------------------------------------------

for column in [
    "Surgery",
    "Chemotherapy",
    "Hormone"
]:

    cleaned_data[column] = (
        cleaned_data[column]
        .apply(normalize_text)
        .fillna("NO")
        .map({
            "YES": "Yes",
            "NO": "No"
        })
    )


# ---------------------------------------------------------
# 17. Pregnancy at diagnosis
# ---------------------------------------------------------

def recode_pregnancy(value):

    text = normalize_text(value)

    if text in {"YES", "POSITIVE"}:
        return "Yes"

    if text in {"NO", "NEGATIVE"}:
        return "No"

    return "Unknown"


cleaned_data["Pregnancy_Status_Clean"] = (
    cleaned_data["Pregnant at Diagnosis"]
    .apply(recode_pregnancy)
)


# ---------------------------------------------------------
# 18. Reproductive numerical variables
# ---------------------------------------------------------

def recode_numeric_with_no_as_zero(value):

    text = normalize_text(value)

    if text == "NO":
        return 0.0

    if text in {None, "UNKNOWN"}:
        return np.nan

    return pd.to_numeric(text, errors="coerce")


cleaned_data["Parity_Numeric"] = (
    cleaned_data["Parity"]
    .apply(recode_numeric_with_no_as_zero)
)

cleaned_data["Abortion_Count"] = (
    cleaned_data["Number of Abortion"]
    .apply(recode_numeric_with_no_as_zero)
)


def recode_optional_age(value):

    text = normalize_text(value)

    if text in {None, "UNKNOWN", "NO"}:
        return np.nan

    return pd.to_numeric(text, errors="coerce")


cleaned_data["Age_First_Childbirth"] = (
    cleaned_data["Age at First Child Birth"]
    .apply(recode_optional_age)
)

cleaned_data["Menarche_Age"] = (
    cleaned_data["Menarche Age"]
    .apply(recode_optional_age)
)


# ---------------------------------------------------------
# 19. Menopause status
# ---------------------------------------------------------

def recode_menopause(row):

    gender = normalize_text(row["Gender"])
    status = normalize_text(
        row["Menopause Status (Y/N, UNKNOWN)"]
    )

    if gender == "MALE":
        return "Not_Applicable"

    if status == "POST":
        return "Post"

    if status in {"PRE", "NO"}:
        return "Pre"

    return "Unknown"


cleaned_data["Menopause_Status_Clean"] = (
    cleaned_data.apply(
        recode_menopause,
        axis=1
    )
)


# ---------------------------------------------------------
# 20. Define the primary modelling dataset
# ---------------------------------------------------------

primary_predictors = [
    "Age_at_Diagnosis",
    "Marital_Status",
    "Nationality_Clean",
    "Primary_Site_Group",
    "Laterality_Clean",
    "Histology_Group",
    "Combined_Grade",
    "Tumor_Size_mm",
    "Summary_Stage_Clean",
    "ER_Status",
    "PR_Status",
    "HER2_Status",
    "Surgery",
    "Chemotherapy",
    "Hormone",
    "Menopause_Status_Clean",
    "Family_History_Breast_Cancer"
]


modeling_columns = (
    ["Source_Excel_Row"]
    + primary_predictors
    + ["Five_Year_Recurrence"]
)


modeling_dataset = cleaned_data[
    modeling_columns
].copy()


# ---------------------------------------------------------
# 21. Validation checks
# ---------------------------------------------------------

assert len(cleaned_data) == 1370
assert len(modeling_dataset) == 1370

assert modeling_dataset[
    "Five_Year_Recurrence"
].eq("Yes").sum() == 73

assert modeling_dataset[
    "Five_Year_Recurrence"
].eq("No").sum() == 1297

assert cleaned_data[
    "Combined_Grade"
].ne("Unknown").sum() == 510

assert cleaned_data[
    "Family_History_Breast_Cancer"
].eq("Yes").sum() == 81

assert cleaned_data[
    "Family_History_Other_Cancer"
].eq("Yes").sum() == 43

assert cleaned_data[
    "Tumor_Size_mm"
].isna().sum() == 174

assert cleaned_data[
    "Stage_Metastasis_Disagreement"
].eq("Yes").sum() == 16

assert cleaned_data[
    "Surgery"
].eq("No").sum() == 360

assert cleaned_data[
    "Chemotherapy"
].eq("No").sum() == 427

assert cleaned_data[
    "Hormone"
].eq("No").sum() == 729


print("All cleaning validation checks passed.")

print(
    "\nCleaned cohort shape:",
    cleaned_data.shape
)

print(
    "Primary modelling dataset shape:",
    modeling_dataset.shape
)

print(
    "Number of primary predictors:",
    len(primary_predictors)
)

print("\nPrimary predictors:")

for number, predictor in enumerate(
    primary_predictors,
    start=1
):
    print(f"{number:2}. {predictor}")

In [ ]:
cleaned_cohort_path = (
    PROCESSED_DATA_DIRECTORY
    / "cleaned_analysis_cohort.xlsx"
)

modeling_dataset_path = (
    PROCESSED_DATA_DIRECTORY
    / "primary_modeling_dataset.xlsx"
)


cleaned_data.to_excel(
    cleaned_cohort_path,
    sheet_name="Cleaned_Cohort",
    index=False,
    engine="openpyxl"
)

modeling_dataset.to_excel(
    modeling_dataset_path,
    sheet_name="Modeling_Dataset",
    index=False,
    engine="openpyxl"
)


print("Files saved successfully:")
print(cleaned_cohort_path)
print(modeling_dataset_path)

In [ ]:
# ============================================================
# Load the primary modelling dataset
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd


PROCESSED_DATA_DIRECTORY = PROJECT_ROOT / "Data" / "Processed"

PRIMARY_MODELING_FILE = (
    PROCESSED_DATA_DIRECTORY
    / "primary_modeling_dataset.xlsx"
)


if not PRIMARY_MODELING_FILE.exists():
    raise FileNotFoundError(
        "The modelling dataset was not found:\n"
        f"{PRIMARY_MODELING_FILE}"
    )


modeling_data = pd.read_excel(
    PRIMARY_MODELING_FILE
)


print("Primary modelling dataset loaded successfully.")
print(f"File: {PRIMARY_MODELING_FILE}")

In [ ]:
# ============================================================
# 1. Dataset dimensions
# 2. Exact variable names
# ============================================================

number_of_rows, number_of_columns = modeling_data.shape


print("DATASET DIMENSIONS")
print("------------------")
print("Number of rows:", number_of_rows)
print("Number of columns:", number_of_columns)


print("\nEXACT VARIABLE NAMES")
print("--------------------")

for position, column in enumerate(
    modeling_data.columns,
    start=1
):
    print(f"{position}. {column}")

In [ ]:
# ============================================================
# Inspect data types and sample records
# ============================================================

variable_information = pd.DataFrame({
    "Variable": modeling_data.columns,
    "Data_Type": modeling_data.dtypes.astype(str).values,
    "Non_Missing_Count": modeling_data.notna().sum().values,
    "Unique_Value_Count": [
        modeling_data[column].nunique(dropna=True)
        for column in modeling_data.columns
    ]
})


print("VARIABLE INFORMATION")
display(variable_information)


print("\nFIRST FIVE RECORDS")
display(modeling_data.head())

In [ ]:
# ============================================================
# 3. Outcome distribution
# ============================================================

OUTCOME_COLUMN = "Five_Year_Recurrence"


if OUTCOME_COLUMN not in modeling_data.columns:
    raise KeyError(
        f"The expected outcome column '{OUTCOME_COLUMN}' "
        "was not found.\n"
        "Review the variable-name output from Cell 2."
    )


outcome_distribution = (
    modeling_data[OUTCOME_COLUMN]
    .value_counts(dropna=False)
    .rename_axis(OUTCOME_COLUMN)
    .reset_index(name="Count")
)


outcome_distribution["Percentage"] = (
    outcome_distribution["Count"]
    / len(modeling_data)
    * 100
).round(2)


print("OUTCOME DISTRIBUTION")
display(outcome_distribution)


print(
    "Missing outcome values:",
    int(modeling_data[OUTCOME_COLUMN].isna().sum())
)

In [ ]:
# ============================================================
# 4. Missing-value inspection
# ============================================================

missing_value_summary = pd.DataFrame({
    "Variable": modeling_data.columns,
    "Missing_Count": modeling_data.isna().sum().values
})


missing_value_summary["Missing_Percentage"] = (
    missing_value_summary["Missing_Count"]
    / len(modeling_data)
    * 100
).round(2)


missing_value_summary = (
    missing_value_summary
    .sort_values(
        by=[
            "Missing_Count",
            "Variable"
        ],
        ascending=[
            False,
            True
        ]
    )
    .reset_index(drop=True)
)


print("STANDARD MISSING VALUES")
display(missing_value_summary)


print(
    "\nVariables containing standard missing values:",
    int((missing_value_summary["Missing_Count"] > 0).sum())
)

In [ ]:
# ============================================================
# Check explicit Unknown categories
# ============================================================

unknown_summary_records = []


for column in modeling_data.columns:

    text_values = (
        modeling_data[column]
        .astype("string")
        .str.strip()
        .str.upper()
    )

    unknown_count = int(
        text_values.isin({
            "UNKNOWN",
            "NOT KNOWN",
            "NOT AVAILABLE",
            "NOT RECORDED",
            "(NONE)"
        }).sum()
    )

    if unknown_count > 0:
        unknown_summary_records.append({
            "Variable": column,
            "Unknown_Count": unknown_count,
            "Unknown_Percentage": round(
                unknown_count
                / len(modeling_data)
                * 100,
                2
            )
        })


unknown_category_summary = pd.DataFrame(
    unknown_summary_records
)


print("EXPLICIT UNKNOWN CATEGORIES")

if unknown_category_summary.empty:
    print("No explicit Unknown categories were found.")
else:
    unknown_category_summary = (
        unknown_category_summary
        .sort_values(
            by="Unknown_Count",
            ascending=False
        )
        .reset_index(drop=True)
    )

    display(unknown_category_summary)

In [ ]:
# ============================================================
# 5. Categorical-variable distributions
# ============================================================

categorical_columns = modeling_data.select_dtypes(
    include=[
        "object",
        "category",
        "string",
        "bool"
    ]
).columns.tolist()


print(
    "Number of categorical variables:",
    len(categorical_columns)
)


for column in categorical_columns:

    category_summary = (
        modeling_data[column]
        .value_counts(dropna=False)
        .rename_axis(column)
        .reset_index(name="Count")
    )

    category_summary["Percentage"] = (
        category_summary["Count"]
        / len(modeling_data)
        * 100
    ).round(2)

    print("\n" + "=" * 70)
    print(f"CATEGORICAL VARIABLE: {column}")
    print("=" * 70)

    display(category_summary)

In [ ]:
# ============================================================
# 6. Numeric-variable ranges
# ============================================================

numeric_columns = modeling_data.select_dtypes(
    include=np.number
).columns.tolist()


numeric_range_summary = pd.DataFrame({
    "Variable": numeric_columns,
    "Non_Missing_Count": [
        modeling_data[column].notna().sum()
        for column in numeric_columns
    ],
    "Missing_Count": [
        modeling_data[column].isna().sum()
        for column in numeric_columns
    ],
    "Minimum": [
        modeling_data[column].min()
        for column in numeric_columns
    ],
    "First_Quartile": [
        modeling_data[column].quantile(0.25)
        for column in numeric_columns
    ],
    "Median": [
        modeling_data[column].median()
        for column in numeric_columns
    ],
    "Mean": [
        modeling_data[column].mean()
        for column in numeric_columns
    ],
    "Third_Quartile": [
        modeling_data[column].quantile(0.75)
        for column in numeric_columns
    ],
    "Maximum": [
        modeling_data[column].max()
        for column in numeric_columns
    ]
})


print("NUMERIC-VARIABLE SUMMARY")
display(numeric_range_summary)

In [ ]:
# ============================================================
# Focused inspection of age and tumour-size variables
# ============================================================

age_candidates = [
    column
    for column in modeling_data.columns
    if "age" in column.lower()
]

tumour_size_candidates = [
    column
    for column in modeling_data.columns
    if (
        "tumour" in column.lower()
        or "tumor" in column.lower()
    )
    and "size" in column.lower()
]


print("Possible age variables:")
print(age_candidates)


print("\nPossible tumour-size variables:")
print(tumour_size_candidates)


for column in age_candidates + tumour_size_candidates:

    numeric_values = pd.to_numeric(
        modeling_data[column],
        errors="coerce"
    )

    print("\n" + "=" * 70)
    print(f"NUMERIC INSPECTION: {column}")
    print("=" * 70)

    print(numeric_values.describe())

    print(
        "\nNumber converted to missing during "
        "numeric inspection:",
        int(
            modeling_data[column].notna().sum()
            - numeric_values.notna().sum()
        )
    )

    print("\nTen smallest values:")
    display(
        numeric_values
        .dropna()
        .sort_values()
        .head(10)
        .to_frame(name=column)
    )

    print("\nTen largest values:")
    display(
        numeric_values
        .dropna()
        .sort_values(ascending=False)
        .head(10)
        .to_frame(name=column)
    )

In [ ]:
# ============================================================
# 7. Check for completely duplicated records
# ============================================================

complete_duplicate_mask = modeling_data.duplicated(
    keep=False
)


number_of_complete_duplicate_rows = int(
    complete_duplicate_mask.sum()
)


print(
    "Number of rows belonging to complete duplicate groups:",
    number_of_complete_duplicate_rows
)


if number_of_complete_duplicate_rows > 0:

    complete_duplicate_records = (
        modeling_data.loc[
            complete_duplicate_mask
        ]
        .sort_values(
            by=modeling_data.columns.tolist()
        )
    )

    display(complete_duplicate_records)

else:
    print("No completely duplicated rows were found.")

In [ ]:
# ============================================================
# Identify possible patient identifier columns
# ============================================================

identifier_keywords = [
    "patient",
    "record",
    "registry",
    "hospital",
    "medical",
    "folder",
    "case",
    "source_excel_row",
    "source row",
    "source_row",
    "id"
]


possible_identifier_columns = [
    column
    for column in modeling_data.columns
    if any(
        keyword in column.lower()
        for keyword in identifier_keywords
    )
]


print("Possible patient identifier columns:")

if possible_identifier_columns:
    for column in possible_identifier_columns:
        print(
            f"- {column}: "
            f"{modeling_data[column].nunique(dropna=False)} "
            "unique values"
        )
else:
    print(
        "No obvious patient identifier column was found "
        "in the modelling dataset."
    )

In [ ]:
# ============================================================
# 8. Screen variable names for possible outcome leakage
# ============================================================

leakage_keywords = [
    "recurrence",
    "relapse",
    "outcome",
    "event",
    "follow",
    "survival",
    "death",
    "dead",
    "vital",
    "status",
    "last_contact",
    "last contact",
    "date_of_death",
    "date of death",
    "recurrence_date",
    "recurrence date",
    "disease_free",
    "disease free"
]


possible_leakage_variables = [
    column
    for column in modeling_data.columns
    if column != OUTCOME_COLUMN
    and any(
        keyword in column.lower()
        for keyword in leakage_keywords
    )
]


print("POSSIBLE OUTCOME-LEAKAGE VARIABLES")


if possible_leakage_variables:

    for column in possible_leakage_variables:
        print(f"- {column}")

else:
    print(
        "No additional variables were flagged by "
        "the leakage-name screening."
    )

In [ ]:
# ============================================================
# Final predictor-name review
# ============================================================

predictor_columns = [
    column
    for column in modeling_data.columns
    if column != OUTCOME_COLUMN
]


print("OUTCOME VARIABLE")
print("----------------")
print(OUTCOME_COLUMN)


print("\nPROPOSED PREDICTOR VARIABLES")
print("----------------------------")

for position, column in enumerate(
    predictor_columns,
    start=1
):
    print(f"{position}. {column}")


print(
    "\nNumber of proposed predictors:",
    len(predictor_columns)
)

In [ ]:
# ============================================================
# Inspect extreme age and tumour-size records
# ============================================================

age_review = (
    modeling_data.loc[
        modeling_data["Age_at_Diagnosis"] >= 100,
        [
            "Source_Excel_Row",
            "Age_at_Diagnosis",
            "Five_Year_Recurrence"
        ]
    ]
    .sort_values(
        by="Age_at_Diagnosis",
        ascending=False
    )
    .reset_index(drop=True)
)


tumour_size_review = (
    modeling_data.loc[
        modeling_data["Tumor_Size_mm"] >= 150,
        [
            "Source_Excel_Row",
            "Tumor_Size_mm",
            "Primary_Site_Group",
            "Summary_Stage_Clean",
            "Five_Year_Recurrence"
        ]
    ]
    .sort_values(
        by="Tumor_Size_mm",
        ascending=False
    )
    .reset_index(drop=True)
)


print("PATIENTS AGED 100 YEARS OR OLDER")
display(age_review)


print("\nPATIENTS WITH TUMOUR SIZE OF AT LEAST 150 MM")
display(tumour_size_review)

In [ ]:
# ============================================================
# Recurrence distribution for every categorical predictor
# ============================================================

OUTCOME_COLUMN = "Five_Year_Recurrence"

categorical_predictors = [
    column
    for column in modeling_data.select_dtypes(
        include=[
            "object",
            "category",
            "string",
            "bool"
        ]
    ).columns
    if column != OUTCOME_COLUMN
]


for column in categorical_predictors:

    outcome_table = pd.crosstab(
        modeling_data[column],
        modeling_data[OUTCOME_COLUMN],
        margins=True
    )

    print("\n" + "=" * 70)
    print(f"{column} BY FIVE-YEAR RECURRENCE")
    print("=" * 70)

    display(outcome_table)

In [ ]:
# ============================================================
# Verify construction of Primary_Site_Group
# ============================================================

site_columns = [
    column
    for column in cleaned_data.columns
    if "site" in column.lower()
]


print("Columns containing the word 'site':")
print(site_columns)

In [ ]:
ORIGINAL_SITE_COLUMN = "Site - Primary (ICD-O-3)"


site_grouping_audit = pd.crosstab(
    cleaned_data[ORIGINAL_SITE_COLUMN],
    cleaned_data["Primary_Site_Group"],
    margins=True,
    dropna=False
)


print("ORIGINAL SITE VERSUS GROUPED SITE")
display(site_grouping_audit)

In [ ]:
# ============================================================
# Create the final, clearly named modelling dataset
# Prediction time: after initial treatment
# ============================================================

final_modeling_data = modeling_data.copy()


# ------------------------------------------------------------
# Rename variables clearly
# ------------------------------------------------------------

variable_rename_map = {
    "Nationality_Clean": "Nationality",
    "Laterality_Clean": "Laterality",
    "Combined_Grade": "Tumor_Grade",
    "Summary_Stage_Clean": "Summary_Stage",
    "Hormone": "Hormone_Therapy",
    "Menopause_Status_Clean": "Menopause_Status",
    "Family_History_Breast_Cancer": (
        "Breast_Cancer_Family_History"
    )
}


final_modeling_data = final_modeling_data.rename(
    columns=variable_rename_map
)


# ------------------------------------------------------------
# Remove the audit identifier from the predictor dataset
# ------------------------------------------------------------

final_modeling_data = final_modeling_data.drop(
    columns=["Source_Excel_Row"]
)


# ------------------------------------------------------------
# Arrange variables in a logical order
# ------------------------------------------------------------

final_variable_order = [
    "Age_at_Diagnosis",
    "Marital_Status",
    "Nationality",
    "Menopause_Status",
    "Breast_Cancer_Family_History",
    "Primary_Site_Group",
    "Laterality",
    "Histology_Group",
    "Tumor_Grade",
    "Tumor_Size_mm",
    "Summary_Stage",
    "ER_Status",
    "PR_Status",
    "HER2_Status",
    "Surgery",
    "Chemotherapy",
    "Hormone_Therapy",
    "Five_Year_Recurrence"
]


missing_expected_columns = [
    column
    for column in final_variable_order
    if column not in final_modeling_data.columns
]


unexpected_columns = [
    column
    for column in final_modeling_data.columns
    if column not in final_variable_order
]


if missing_expected_columns:
    raise KeyError(
        "Expected variables are missing:\n"
        + "\n".join(missing_expected_columns)
    )


if unexpected_columns:
    raise KeyError(
        "Unexpected variables were found:\n"
        + "\n".join(unexpected_columns)
    )


final_modeling_data = final_modeling_data[
    final_variable_order
].copy()


# ------------------------------------------------------------
# Validate dimensions and outcome
# ------------------------------------------------------------

assert len(final_modeling_data) == 1370, (
    "Expected 1,370 patient records."
)

assert final_modeling_data.shape[1] == 18, (
    "Expected 17 candidate predictors and one outcome."
)

assert (
    final_modeling_data["Five_Year_Recurrence"]
    .isna()
    .sum()
    == 0
), "The outcome must not contain missing values."

assert (
    final_modeling_data["Five_Year_Recurrence"]
    .value_counts()
    .to_dict()
    == {
        "No": 1297,
        "Yes": 73
    }
), "The recurrence distribution has changed unexpectedly."

assert (
    final_modeling_data.duplicated().sum()
    == 0
), "Completely duplicated records were detected."

assert (
    final_modeling_data["Tumor_Size_mm"]
    .isna()
    .sum()
    == 174
), "The expected tumour-size missing count has changed."

assert (
    final_modeling_data["Age_at_Diagnosis"].max()
    == 103
), "The maximum verified age has changed."

assert (
    final_modeling_data["Tumor_Size_mm"].max()
    == 270
), "The maximum verified tumour size has changed."


# ------------------------------------------------------------
# Display final structure
# ------------------------------------------------------------

print("FINAL MODELLING DATASET")
print("-----------------------")
print("Number of patients:", final_modeling_data.shape[0])
print(
    "Number of candidate predictors:",
    final_modeling_data.shape[1] - 1
)
print("Number of outcome variables: 1")


print("\nFINAL VARIABLE NAMES")

for position, column in enumerate(
    final_modeling_data.columns,
    start=1
):
    print(f"{position}. {column}")


print("\nOUTCOME DISTRIBUTION")

display(
    final_modeling_data[
        "Five_Year_Recurrence"
    ].value_counts(dropna=False)
)


print("\nSTANDARD MISSING VALUES")

display(
    final_modeling_data
    .isna()
    .sum()
    .sort_values(ascending=False)
    .to_frame(name="Missing_Count")
)


print("\nFIRST FIVE RECORDS")

display(
    final_modeling_data.head()
)

In [ ]:
# ============================================================
# Save the final modelling dataset
# 17 candidate predictors plus one outcome
# ============================================================

FINAL_MODELING_FILE = (
    PROCESSED_DATA_DIRECTORY
    / "final_modeling_dataset.xlsx"
)


final_modeling_data.to_excel(
    FINAL_MODELING_FILE,
    index=False
)


print("Final modelling dataset saved successfully:")
print(FINAL_MODELING_FILE)

In [ ]:
# ============================================================
# Reload and verify the saved final modelling dataset
# ============================================================

verified_final_data = pd.read_excel(
    FINAL_MODELING_FILE
)


expected_final_columns = [
    "Age_at_Diagnosis",
    "Marital_Status",
    "Nationality",
    "Menopause_Status",
    "Breast_Cancer_Family_History",
    "Primary_Site_Group",
    "Laterality",
    "Histology_Group",
    "Tumor_Grade",
    "Tumor_Size_mm",
    "Summary_Stage",
    "ER_Status",
    "PR_Status",
    "HER2_Status",
    "Surgery",
    "Chemotherapy",
    "Hormone_Therapy",
    "Five_Year_Recurrence"
]


assert verified_final_data.shape == (1370, 18), (
    "Expected 1,370 rows and 18 columns."
)

assert (
    verified_final_data.columns.tolist()
    == expected_final_columns
), "The saved variables or their order are incorrect."

assert (
    "Source_Excel_Row"
    not in verified_final_data.columns
), "Source_Excel_Row must not appear in the final file."

assert (
    verified_final_data["Five_Year_Recurrence"]
    .value_counts()
    .to_dict()
    == {
        "No": 1297,
        "Yes": 73
    }
), "The saved outcome distribution is incorrect."

assert (
    verified_final_data["Tumor_Size_mm"]
    .isna()
    .sum()
    == 174
), "The tumour-size missing count has changed."

assert (
    verified_final_data.duplicated().sum()
    == 0
), "The final file contains completely duplicated rows."


print("FINAL FILE VERIFICATION PASSED")
print("------------------------------")
print("File:", FINAL_MODELING_FILE)
print("Patients:", verified_final_data.shape[0])
print("Candidate predictors:", verified_final_data.shape[1] - 1)
print("Outcome variables: 1")
print("Tumour-size missing values:", end=" ")
print(
    verified_final_data[
        "Tumor_Size_mm"
    ].isna().sum()
)


print("\nVerified variables:")

for position, column in enumerate(
    verified_final_data.columns,
    start=1
):
    print(f"{position}. {column}")

In [ ]:
# ============================================================
# Rename the final outcome variable to Recurrence
# ============================================================

final_modeling_data = pd.read_excel(
    FINAL_MODELING_FILE
)


# Rename only if the previous name is present.
if "Five_Year_Recurrence" in final_modeling_data.columns:

    final_modeling_data = (
        final_modeling_data.rename(
            columns={
                "Five_Year_Recurrence": "Recurrence"
            }
        )
    )


expected_final_columns = [
    "Age_at_Diagnosis",
    "Marital_Status",
    "Nationality",
    "Menopause_Status",
    "Breast_Cancer_Family_History",
    "Primary_Site_Group",
    "Laterality",
    "Histology_Group",
    "Tumor_Grade",
    "Tumor_Size_mm",
    "Summary_Stage",
    "ER_Status",
    "PR_Status",
    "HER2_Status",
    "Surgery",
    "Chemotherapy",
    "Hormone_Therapy",
    "Recurrence"
]


assert final_modeling_data.shape == (1370, 18)

assert (
    final_modeling_data.columns.tolist()
    == expected_final_columns
), "The final variables or their order are incorrect."

assert (
    "Five_Year_Recurrence"
    not in final_modeling_data.columns
)

assert (
    final_modeling_data["Recurrence"]
    .value_counts()
    .to_dict()
    == {
        "No": 1297,
        "Yes": 73
    }
), "The recurrence distribution has changed."


# Overwrite the final file with the corrected outcome name.
final_modeling_data.to_excel(
    FINAL_MODELING_FILE,
    index=False,
    engine="openpyxl"
)


# Independently reload and verify the saved workbook.
verified_renamed_data = pd.read_excel(
    FINAL_MODELING_FILE
)


assert (
    verified_renamed_data.columns.tolist()
    == expected_final_columns
)

assert (
    verified_renamed_data["Recurrence"]
    .value_counts()
    .to_dict()
    == {
        "No": 1297,
        "Yes": 73
    }
)


print("FINAL OUTCOME VARIABLE RENAMED SUCCESSFULLY")
print("-------------------------------------------")
print("File:", FINAL_MODELING_FILE)
print("Patients:", verified_renamed_data.shape[0])
print("Candidate predictors:", verified_renamed_data.shape[1] - 1)
print("Outcome variable: Recurrence")

print("\nOutcome distribution:")

display(
    verified_renamed_data[
        "Recurrence"
    ].value_counts(dropna=False)
)